# AI-Enhanced Decision Support Layer

**Author:** Kevin Danh  
**Extension of Earlier Project:** Predictive Maintenance Using Machine Learning

---

## Overview

In Phase 1, we built and validated an XGBoost model capable of flagging machines at risk of failure with a ROC-AUC of 0.76. While the model produces useful risk scores, it does not explain *why* a machine is at risk or *what* a maintenance team should do next.

In Phase 2, we add a Generative AI layer that:
1. Takes the model's predicted probability and top contributing features
2. Constructs a structured prompt
3. Calls the OpenAI API (GPT-4o-mini) to generate a plain-English explanation and maintenance recommendation

This mirrors how AI-enhanced decision support is deployed in real industrial environments — a predictive model surfaces the risk, and a language model translates it into actionable guidance for operators.

In [6]:
from openai import OpenAI
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

In [7]:
load_dotenv()  # loads from .env file in the same folder as your notebook

api_key = os.getenv("OPENAI_KEY")
client = OpenAI(api_key=api_key)

## Load the Trained Model and Test Data

We reload the XGBoost model and test set from Phase 1.

In [20]:
import joblib
xgb    = joblib.load('xgb_model.pkl')
X_test = joblib.load('X_test.pkl')
y_test = joblib.load('y_test.pkl')
y_prob = joblib.load('y_prob.pkl')

# Recompute predicted probabilities (Phase 1 threshold = 0.01)
THRESHOLD = 0.01

print(f"Test set size: {len(X_test):,}")
print(f"Flagged at threshold {THRESHOLD}: {(y_prob >= THRESHOLD).sum()}")
print(f"True failures in test set: {y_test.sum()}")

Test set size: 24,432
Flagged at threshold 0.01: 153
True failures in test set: 21


## Define the AI Explanation Function

This function takes a single device's sensor readings, its predicted failure probability, and the top contributing features, then returns a plain-English explanation and maintenance recommendation from the OpenAI API.

The prompt is structured to:
- Provide model context (so the LLM understands the analytical foundation)
- Supply only the most influential features (avoids prompt noise)
- Request a specific output format: condition summary, recommended action, urgency level

In [13]:
TOP_FEATURES = ['metric4_roll3', 'metric2_lag1', 'metric3_lag1', 'metric7_lag1', 'metric4_log']

def build_prompt(device_id, row, prob, top_features=TOP_FEATURES):
    """
    Build a structured maintenance prompt from a device's sensor state.

    Parameters
    ----------
    device_id : str
        Device identifier for context.
    row : pd.Series
        Feature row for the device at this timestamp.
    prob : float
        Predicted failure probability from the XGBoost model.
    top_features : list
        Feature names to include in the prompt (from importance ranking).

    Returns
    -------
    str
        Formatted prompt string.
    """
    # Only include features that exist in this row
    available = [f for f in top_features if f in row.index]
    feat_lines = "\n".join([f"  - {f}: {row[f]:.4f}" for f in available])

    urgency = "LOW" if prob < 0.20 else "MEDIUM" if prob < 0.50 else "HIGH"

    prompt = f"""You are an industrial predictive maintenance assistant.

A trained XGBoost classifier (ROC-AUC: 0.76, decision threshold: 0.01) has flagged device {device_id} with a {prob:.1%} estimated failure probability.

The model's top contributing sensor signals for this prediction:
{feat_lines}

Feature notes:
- metric4_roll3: 3-day rolling average of metric4 (highest importance feature in the model)
- metric2_lag1: metric2 reading from the previous day
- metric3_lag1: metric3 reading from the previous day
- metric7_lag1: metric7 reading from the previous day
- metric4_log: log-transformed current reading of metric4

Note: sensor metric labels are anonymized. Interpretations are based on statistical 
patterns in the data rather than known physical measurements.

Provide a maintenance report with exactly three sections:
1. CONDITION SUMMARY: 2-3 sentences interpreting what these sensor patterns suggest about the device's current state based on statistical anomalies relative to normal operating ranges.
2. RECOMMENDED ACTION: 1-2 concrete maintenance steps a technician should take.
3. URGENCY: {urgency} — briefly justify this urgency level in one sentence.
"""
    return prompt


def get_ai_explanation(device_id, row, prob, top_features=TOP_FEATURES):
    """
    Call the OpenAI API and return the maintenance explanation.
    """
    prompt = build_prompt(device_id, row, prob, top_features)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

## Select Representative Cases

To illustrate the AI layer across different scenarios, we select four cases:

| Case | Description | Why It's Useful |
|------|-------------|------------------|
| True Positive | Model flagged a device that actually failed | Shows the system working as intended |
| False Positive | Model flagged a device that did not fail | Demonstrates honest system limitations |
| High Probability | Highest-scored device in test set | Stress test of the explanation quality |
| Low Probability | Normal operating device | Confirms the system doesn't over-alarm |

Including all four cases is important for academic credibility — only showing successes misrepresents the model.

In [14]:
# Build a combined dataframe with predictions for easy slicing
results_df = X_test.copy()
results_df['failure_actual'] = y_test.values
results_df['failure_prob'] = y_prob
results_df['flagged'] = (y_prob >= THRESHOLD).astype(int)

# Case 1: True Positive — flagged AND actually failed
true_positives = results_df[(results_df['flagged'] == 1) & (results_df['failure_actual'] == 1)]
case_tp = true_positives.sort_values('failure_prob', ascending=False).iloc[0]

# Case 2: False Positive — flagged but did NOT fail
false_positives = results_df[(results_df['flagged'] == 1) & (results_df['failure_actual'] == 0)]
case_fp = false_positives.sort_values('failure_prob', ascending=False).iloc[0]

# Case 3: Highest probability in test set (regardless of outcome)
case_high = results_df.sort_values('failure_prob', ascending=False).iloc[0]

# Case 4: Low probability — clearly normal device
case_low = results_df[results_df['failure_prob'] < 0.005].iloc[0]

print(f"Case 1 (True Positive)  — Prob: {case_tp['failure_prob']:.4f}, Actual: {int(case_tp['failure_actual'])}")
print(f"Case 2 (False Positive) — Prob: {case_fp['failure_prob']:.4f}, Actual: {int(case_fp['failure_actual'])}")
print(f"Case 3 (High Risk)      — Prob: {case_high['failure_prob']:.4f}, Actual: {int(case_high['failure_actual'])}")
print(f"Case 4 (Low Risk)       — Prob: {case_low['failure_prob']:.4f}, Actual: {int(case_low['failure_actual'])}")

Case 1 (True Positive)  — Prob: 0.6206, Actual: 1
Case 2 (False Positive) — Prob: 0.9714, Actual: 0
Case 3 (High Risk)      — Prob: 0.9714, Actual: 0
Case 4 (Low Risk)       — Prob: 0.0000, Actual: 0


## Case 1: True Positive

The model correctly identified a device that subsequently failed. This is the core use case — early detection of a real failure event.

In [15]:
print("=" * 60)
print("CASE 1: TRUE POSITIVE")
print(f"Predicted probability: {case_tp['failure_prob']:.4f}")
print(f"Actual outcome: FAILURE")
print("=" * 60)
print()

explanation_tp = get_ai_explanation(
    device_id="Device_TP",
    row=case_tp,
    prob=case_tp['failure_prob']
)

print(explanation_tp)

CASE 1: TRUE POSITIVE
Predicted probability: 0.6206
Actual outcome: FAILURE

### CONDITION SUMMARY:
The sensor patterns indicate significant anomalies, notably a maximal reading in metric4_roll3 alongside elevated values in metric2_lag1. The absence of recent readings in both metric3_lag1 and metric7_lag1, combined with the log-transformed current reading of metric4, suggests that Device_TP is experiencing abnormal operational conditions which could lead to potential failure.

### RECOMMENDED ACTION:
A technician should conduct a thorough inspection of the device, focusing particularly on the mechanisms influenced by metric4, as well as the overall health of the sensors related to this metric. Additionally, a review and potential recalibration of the device associated with metric2 may also be necessary.

### URGENCY: HIGH  
This urgency level is justified due to the estimated failure probability of 62.1%, indicating a significant risk of failure that requires immediate attention to pre

## Case 2: False Positive

The model flagged this device, but no failure occurred. This case is included deliberately — it shows an honest limitation of the system and demonstrates how the AI explanation can still provide value by identifying genuinely abnormal readings worth monitoring, even if they did not lead to failure in this instance.

In predictive maintenance, false positives are operationally preferable to missed failures (false negatives), as the cost of unnecessary inspection is far lower than unplanned downtime.

In [16]:
print("=" * 60)
print("CASE 2: FALSE POSITIVE")
print(f"Predicted probability: {case_fp['failure_prob']:.4f}")
print(f"Actual outcome: NO FAILURE")
print("=" * 60)
print()

explanation_fp = get_ai_explanation(
    device_id="Device_FP",
    row=case_fp,
    prob=case_fp['failure_prob']
)

print(explanation_fp)

CASE 2: FALSE POSITIVE
Predicted probability: 0.9714
Actual outcome: NO FAILURE

### CONDITION SUMMARY:
The sensor patterns indicate significant deviations from normal operating ranges, particularly highlighted by a high rolling average of metric4 and an elevated lagged value of metric2. The combination of these anomalies suggests that Device_FP is experiencing severe operational stress, which strongly correlates with a high failure probability.

### RECOMMENDED ACTION:
The technician should conduct an immediate thorough inspection of Device_FP, focusing on the components related to metric4 and metric2. Additionally, replacing any worn or malfunctioning parts indicated by these sensor readings is advised to mitigate the risk of failure.

### URGENCY:
HIGH — The device's 97.1% estimated failure probability necessitates immediate attention to prevent unexpected downtime and potential safety hazards.


## Case 3: Highest Risk Device

The device with the highest predicted failure probability in the entire test set. This stress-tests the explanation quality at the extreme end of the probability distribution.

In [17]:
print("=" * 60)
print("CASE 3: HIGHEST PREDICTED RISK")
print(f"Predicted probability: {case_high['failure_prob']:.4f}")
print(f"Actual outcome: {'FAILURE' if case_high['failure_actual'] == 1 else 'NO FAILURE'}")
print("=" * 60)
print()

explanation_high = get_ai_explanation(
    device_id="Device_HIGH",
    row=case_high,
    prob=case_high['failure_prob']
)

print(explanation_high)

CASE 3: HIGHEST PREDICTED RISK
Predicted probability: 0.9714
Actual outcome: NO FAILURE

### CONDITION SUMMARY:
The sensor patterns indicate a critical deviation from normal operating conditions, particularly with metric4_roll3, which is significantly elevated, suggesting an unusual build-up or anomaly that may compromise the device’s performance. Additionally, the combination of other lagged readings shows potential inconsistencies that point towards progressive degradation or impending failure.

### RECOMMENDED ACTION:
Technicians should conduct a thorough inspection of Device_HIGH to identify any mechanical or software issues, and immediately replace or service components associated with metric4 to mitigate potential failure risks.

### URGENCY: HIGH  
Given the 97.1% estimated failure probability, immediate action is essential to prevent catastrophic breakdown and operational downtime.


## Case 4: Normal Operating Device

A device with very low predicted failure probability, representing normal operation. Including this case confirms the system does not over-alarm — it should produce a calm, reassuring explanation for healthy devices.

In [18]:
print("=" * 60)
print("CASE 4: LOW RISK (NORMAL OPERATION)")
print(f"Predicted probability: {case_low['failure_prob']:.4f}")
print(f"Actual outcome: NO FAILURE")
print("=" * 60)
print()

explanation_low = get_ai_explanation(
    device_id="Device_LOW",
    row=case_low,
    prob=case_low['failure_prob']
)

print(explanation_low)

CASE 4: LOW RISK (NORMAL OPERATION)
Predicted probability: 0.0000
Actual outcome: NO FAILURE

**CONDITION SUMMARY:** The sensor signals for Device_LOW indicate an absence of activity, as all the top contributing features show a value of 0.0. This suggests that the device is currently in a non-operational or inactive state, which is consistent with a low estimated failure probability according to the predictive model.

**RECOMMENDED ACTION:** A technician should perform a visual inspection of Device_LOW to ensure it is properly powered and has not been shut down unintentionally. Additionally, check the connectivity of the sensors to confirm that they are functioning correctly and transmitting data.

**URGENCY:** LOW — the low failure probability combined with the lack of anomalies in sensor data suggests that immediate intervention is not required at this time.


## Batch Explanation for Flagged Devices

In a production deployment, you would run this across all flagged devices at each monitoring interval. The cell below generates explanations for the top 5 highest-risk devices and assembles them into a structured maintenance report.

In [19]:
# Get top 5 highest-risk flagged devices
top_flagged = results_df[results_df['flagged'] == 1].sort_values('failure_prob', ascending=False).head(5)

print(f"Generating maintenance report for {len(top_flagged)} highest-risk devices...\n")
print("=" * 60)

report_entries = []

for i, (idx, row) in enumerate(top_flagged.iterrows(), 1):
    device_id = f"Device_{i:03d} (index {idx})"
    prob = row['failure_prob']
    actual = int(row['failure_actual'])

    print(f"\n[{i}/5] {device_id}")
    print(f"  Predicted probability: {prob:.4f} | Actual failure: {actual}")
    print()

    explanation = get_ai_explanation(
        device_id=device_id,
        row=row,
        prob=prob
    )

    print(explanation)
    print("-" * 60)

    report_entries.append({
        'device': device_id,
        'prob': prob,
        'actual': actual,
        'explanation': explanation
    })

print("\nBatch report complete.")

Generating maintenance report for 5 highest-risk devices...


[1/5] Device_001 (index 61993)
  Predicted probability: 0.9714 | Actual failure: 0

### CONDITION SUMMARY:
Device_001 is exhibiting significant anomalies in its sensor signals, notably a high rolling average of metric4 and abnormally elevated readings of metric2, which suggest deteriorating performance or imminent failure. The present values of the contributing metrics indicate that the device is operating well outside of its normal parameters, which typically correlates with potential failure modes.

### RECOMMENDED ACTION:
Technicians should immediately conduct a thorough inspection of Device_001, focusing on the components correlated with metric4 and metric2. Additionally, a detailed diagnostic test should be performed to identify any underlying issues contributing to these abnormal readings.

### URGENCY: HIGH
The 97.1% estimated failure probability reflects an imminent risk of failure, necessitating prompt action to pre

## Observation from Batch Report and Cases 2/3
Since the dataset was originally highly unbalanced or contain a rare number of 106 failures across 124k records this means that the model flags aggressively at low thresholds. Thus, the predicted probability of failure for the devices were high and produced more false positives in exchanged for fewer misses failures. The batch report reflects this tradeoff: all 5 flagged devices show high predicted probabilities but no actual failure occurred.

This is not a model failure--it is the expected behavior of a recall-optimized classifier on a rare-event problem. In predictive maintenance, the cost of an unnecessary inspection is far lower than the cost of missing a real failure and experiencing unplanned downtime. The AI explanation layer adds value here by giving technicians enough context to make an informed judgment call, rather than acting blindly on a probability score alone.

## Prompt Engineering Notes

The prompt design reflects several deliberate choices relevant to AI-enhanced industrial systems:

**Grounding the model in the analytical context.** Providing the ROC-AUC score and threshold explicitly tells the LLM the confidence level of the underlying prediction. This prevents over-confident language when the model's discriminative ability is modest.

**Feature annotation.** Raw feature names like metric4_roll3 are opaque. Adding brief descriptions ("3-day rolling average of metric4") allows the LLM to reason about temporal patterns rather than just numeric values, producing more contextual explanations.

**Structured output format.** Specifying three labeled sections (CONDITION SUMMARY, RECOMMENDED ACTION, URGENCY) ensures consistency across devices, which matters when outputs are consumed by downstream systems or displayed in dashboards.

**Urgency pre-classification.** Computing urgency in Python from the probability score and passing it to the LLM — rather than asking the LLM to infer it — keeps the urgency level anchored to the model's output rather than the LLM's interpretation.

## Comparison: Model Output vs. AI-Enhanced Output

The table below summarizes what each layer of the system contributes:

| | XGBoost Model Only | With AI Explanation Layer |
|---|---|---|
| **Output** | Probability score (e.g. 0.034) | Probability + condition summary + action + urgency |
| **Audience** | Data scientists / analysts | Maintenance technicians, operations managers |
| **Actionability** | Low (number only) | High (specific next steps) |
| **Explainability** | Feature importance chart | Natural language per-device reasoning |
| **Scalability** | High | High (batch API calls) |

This two-layer architecture--predictive model + language model--represents a practical pattern for deploying ML in environments where end users are not data scientists.

## Limitations of the AI Layer

**Hallucination risk.** The LLM does not have access to historical failure patterns from the actual dataset. Its mechanical interpretations (e.g. "sustained pressure buildup suggests bearing wear") are plausible but not validated against domain-specific failure records. In a production system, explanations should be reviewed by a domain expert before being acted upon.

**Prompt sensitivity.** Small changes in prompt wording can produce different explanations for the same input. The current prompt was tested on a small number of cases — broader evaluation would be needed before deployment.

**Feature opacity.** The sensor metrics in this dataset are anonymized (metric1–9). A real deployment would benefit from named features with known physical units, enabling more precise and trustworthy explanations.

**Latency and cost.** Each API call adds ~1–3 seconds and token cost. For real-time monitoring at scale, batch processing with asynchronous calls would be necessary.

## Future Extensions

- **SHAP values as prompt input**: Replace raw feature values with SHAP contribution scores for more precise causal framing in the prompt
- **Retrieval-augmented generation (RAG)**: Attach a database of known failure patterns and past maintenance records so the LLM can ground explanations in historical precedent
- **Feedback loop**: Allow technicians to rate explanation quality, using that signal to refine prompts over time
- **Multimodal input**: Incorporate vibration waveform images or thermal camera feeds alongside sensor tabular data